## **Web Crawling and Scrapping 실습**


HANDS ON #2

Setup and Scraping the First 20 Books
This cell installs the necessary tools and grabs the full titles using the title attribute as shown in your instructions.

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# 1. 페이지 1 주소 설정 및 데이터 요청
url = "http://books.toscrape.com/catalogue/page-1.html"
response = requests.get(url)
soup = BeautifulSoup(response.text, 'html.parser')

# 2. 모든 h3 태그 안의 a 태그를 찾고 'title' 속성값만 가져오기
# (화면에 보이는 짧은 텍스트가 아니라 전체 제목을 가져오는 핵심 코드입니다)
titles = [a['title'] for a in soup.select('h3 a')]

# 결과 출력 (리스트 형태 확인)
print("# 결과 출력")
for title in titles:
    print(title)

# 3. 데이터프레임으로 변환
df_titles = pd.DataFrame(titles, columns=['Book Titles'])

# 최종 데이터프레임 확인
df_titles

# 결과 출력
A Light in the Attic
Tipping the Velvet
Soumission
Sharp Objects
Sapiens: A Brief History of Humankind
The Requiem Red
The Dirty Little Secrets of Getting Your Dream Job
The Coming Woman: A Novel Based on the Life of the Infamous Feminist, Victoria Woodhull
The Boys in the Boat: Nine Americans and Their Epic Quest for Gold at the 1936 Berlin Olympics
The Black Maria
Starving Hearts (Triangular Trade Trilogy, #1)
Shakespeare's Sonnets
Set Me Free
Scott Pilgrim's Precious Little Life (Scott Pilgrim #1)
Rip it Up and Start Again
Our Band Could Be Your Life: Scenes from the American Indie Underground, 1981-1991
Olio
Mesaerion: The Best Science Fiction Stories 1800-1849
Libertarianism for Beginners
It's Only the Himalayas


,Book Titles
0,A Light in the Attic
1,Tipping the Velvet
2,Soumission
3,Sharp Objects
4,Sapiens: A Brief History of Humankind
5,The Requiem Red
6,The Dirty Little Secrets of Getting Your Dream...
7,The Coming Woman: A Novel Based on the Life of...
8,The Boys in the Boat: Nine Americans and Their...
9,The Black Maria


Scraping Categories and Cleaning Prices
This cell handles the "10 categories, 10 books each" requirement and converts prices into numbers so we can sort them.**bold text**

In [ ]:
import time

# List of categories from your image
categories = ['travel_2', 'mystery_3', 'historical-fiction_4', 'sequential-art_5',
              'classics_6', 'philosophy_7', 'romance_8', 'womens-fiction_9',
              'fiction_10', 'childrens_11']

all_books = []

for cat in categories:
    cat_url = f"http://books.toscrape.com/catalogue/category/books/{cat}/index.html"
    res = requests.get(cat_url)
    cat_soup = BeautifulSoup(res.text, 'html.parser')

    # Grab the first 10 books in this category
    products = cat_soup.select('.product_pod')[:10]

    for item in products:
        title = item.select_one('h3 a')['title']
        price = item.select_one('.price_color').text
        category_name = cat.split('_')[0].replace('-', ' ').title()

        all_books.append({'Category': category_name, 'Title': title, 'Price': price})

# Create DataFrame
df_books = pd.DataFrame(all_books)

# CLEANING: Remove 'Â£' and convert to numeric
df_books['Price_Numeric'] = df_books['Price'].str.replace('Â£', '').astype(float)

# Show Top 10 Most Expensive
top_10 = df_books.sort_values(by='Price_Numeric', ascending=False).head(10)
print("\nTop 10 Most Expensive Books:")
print(top_10)


Top 10 Most Expensive Books:
              Category                                              Title  \
46            Classics                                            Candide   
51          Philosophy       The Death of Humanity: and the Case for Life   
93           Childrens  The White Cat and the Monk: A Retelling of the...   
70      Womens Fiction  I Had a Nice Time And Other Lies...: How to fi...   
47            Classics                                        Animal Farm   
7               Travel                   A Year in Provence (Provence #1)   
12             Mystery                                The Past Never Ends   
92           Childrens                    The Secret of Dreadwillow Carse   
66             Romance                   Suddenly in Love (Lake Haven #1)   
22  Historical Fiction            A Flight of Arrows (The Pathfinders #2)   

      Price  Price_Numeric  
46  Â£58.63          58.63  
51  Â£58.11          58.11  
93  Â£58.08          58.08  
70  Â£

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

base_url = "http://books.toscrape.com/"

# Your categories dictionary
categories = {
    "Travel": "travel_2",
    "Mystery": "mystery_3",
    "Historical Fiction": "historical-fiction_4",
    "Sequential Art": "sequential-art_5",
    "Classics": "classics_6",
    "Philosophy": "philosophy_7",
    "Romance": "romance_8",
    "Womens Fiction": "womens-fiction_9",
    "Fiction": "fiction_10",
    "Childrens": "childrens_11"
}

result = []

for category, slug in categories.items():
    page = 1
    max_price = 0
    max_title = ""

    while True:
        # Handling the URL structure for page 1 vs others
        if page == 1:
            url = f"{base_url}catalogue/category/books/{slug}/index.html"
        else:
            url = f"{base_url}catalogue/category/books/{slug}/page-{page}.html"

        response = requests.get(url)

        # 1. 종료 조건: 페이지가 없으면(200이 아니면) 탈출
        if response.status_code != 200:
            break

        response.encoding = "utf-8"
        soup = BeautifulSoup(response.text, "html.parser")

        # 2. 책 리스트 찾아오기
        books = soup.find_all("article", class_="product_pod")

        # 만약 해당 페이지에 책이 아예 없으면 탈출
        if not books:
            break

        for book in books:
            # title 속성값 추출
            title = book.find("h3").find("a")["title"]

            # 가격 텍스트 추출
            price_text = book.find("p", class_="price_color").text

            # 핵심: 숫자와 점(.)만 남기고 제거하여 float로 변환
            price = float(re.sub(r"[^\d.]", "", price_text))

            # 3. 최대 가격 비교 및 업데이트 (Max Tracker)
            if price > max_price:
                max_price = price
                max_title = title

        # 다음 페이지로 이동
        page += 1

    # 모든 페이지를 다 본 후, 해당 카테고리에서 가장 비싼 책 저장
    result.append([category, max_title, max_price])

# 결과 데이터프레임 생성
df_category = pd.DataFrame(result, columns=["Category", "Most Expensive Book", "Price"])

print(df_category)

             Category                                Most Expensive Book  \
0              Travel                   A Year in Provence (Provence #1)   
1             Mystery                      Boar Island (Anna Pigeon #19)   
2  Historical Fiction                   The Last Painting of Sara de Vos   
3      Sequential Art                                           El Deafo   
4            Classics                                            Candide   
5          Philosophy       The Death of Humanity: and the Case for Life   
6             Romance                 The Perfect Play (Play by Play #1)   
7      Womens Fiction  I Had a Nice Time And Other Lies...: How to fi...   
8             Fiction                  Last One Home (New Beginnings #1)   
9           Childrens  The White Cat and the Monk: A Retelling of the...   

   Price  
0  56.88  
1  59.48  
2  55.55  
3  57.62  
4  58.63  
5  58.11  
6  59.99  
7  57.36  
8  59.98  
9  58.08  
